# 05 · PII controls with Presidio: detect, transform, and build purpose-specific views

**Objective (20 min):** run Microsoft Presidio's `AnalyzerEngine` (NLP + pattern recognizers) and
`AnonymizerEngine` on synthetic support text, measure what it **misses** and **over-detects**, add a
custom recognizer, then produce different views for different destinations (model, logs, fraud linkage).

The synthetic text contains a person name, an internal customer ID, an e-mail address and an Indian
mobile number. **We will not put the raw text into any evidence artifact.**

In [ ]:
# --- Workshop bootstrap: run this cell first ------------------------------------
# JupyterLab starts every kernel inside the notebook's own folder. Move to the
# toolkit root so shared modules (demo_agent, workshop_utils) import and the
# _evidence/ output paths resolve, no matter where Jupyter was launched from.
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop root:", ROOT)

In [ ]:
import hmac
import json
import re
from hashlib import sha256
from pathlib import Path

import pandas as pd
from IPython.display import display

from workshop_utils import redact_for_logs, require_package, save_json

require_package("presidio-analyzer", "presidio_analyzer")
require_package("presidio-anonymizer", "presidio_anonymizer")

raw_text = "Customer Asha Rao (CUST-48291) can be reached at asha.rao@example.com or +91 9876543210. Return window?"
print(raw_text)

## 1. Minimal offline fallback (a test double)

`workshop_utils.redact_for_logs` is a handful of regexes. It is useful for unit tests and known
identifier shapes — and it is deliberately **not** a PII detector. Note what it does *not* catch: the
person name and the internal customer ID.

In [ ]:
fallback_view = redact_for_logs(raw_text)
print(fallback_view)
assert "asha.rao@example.com" not in fallback_view
assert "9876543210" not in fallback_view
print("Name still present:", "Asha Rao" in fallback_view, "| Customer ID still present:", "CUST-48291" in fallback_view)

## 2. Presidio `AnalyzerEngine` with a small spaCy model + custom recognizers

Presidio combines an NLP engine (here spaCy `en_core_web_sm`, ~12 MB, installed from the lock file
so no `spacy download` step is needed) with a registry of recognizers. We load the predefined
recognizers (`PERSON`, `EMAIL_ADDRESS`, `PHONE_NUMBER`, …) and add two of our own for
enterprise-specific identifiers.

In [ ]:
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

nlp_engine = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
}).create_engine()

registry = RecognizerRegistry(supported_languages=["en"])
registry.load_predefined_recognizers(nlp_engine=nlp_engine, languages=["en"])
registry.add_recognizer(PatternRecognizer(
    supported_entity="CUSTOMER_ID", supported_language="en",
    patterns=[Pattern("customer-id", r"\bCUST-\d{5}\b", 0.95)],
))
registry.add_recognizer(PatternRecognizer(
    supported_entity="IN_PHONE_NUMBER", supported_language="en",
    patterns=[Pattern("india-mobile", r"(?<!\d)(?:\+?91[-\s]?)?[6-9]\d{9}(?!\d)", 0.85)],
    context=["phone", "mobile", "call", "reached"],
))

analyzer = AnalyzerEngine(nlp_engine=nlp_engine, registry=registry, supported_languages=["en"])
print("Recognizers loaded:", len(registry.recognizers))

In [ ]:
def analyze_table(text: str, **kwargs) -> pd.DataFrame:
    results = analyzer.analyze(text=text, language="en", return_decision_process=True, **kwargs)
    return pd.DataFrame([
        {"entity_type": r.entity_type, "start": r.start, "end": r.end, "score": round(r.score, 2),
         "span": text[r.start:r.end], "recognizer": r.analysis_explanation.recognizer if r.analysis_explanation else None}
        for r in sorted(results, key=lambda r: (r.start, -r.score))
    ])

all_hits = analyze_table(raw_text)
display(all_hits)

Two things to notice in the table above:

* **False positives.** The Indian mobile number also matches `UK_NHS` (score 1.0!) and `DATE_TIME`.
  Predefined recognizers were built for other locales; every deployment must decide which entities
  it actually wants.
* **Overlaps.** Several recognizers fire on the same span. The anonymizer resolves overlaps, but the
  *scores* you see are per-recognizer confidences, not probabilities.

So: pass an explicit `entities=[...]` allow-list and a `score_threshold`.

In [ ]:
WANTED = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "IN_PHONE_NUMBER", "CUSTOMER_ID"]
hits = analyze_table(raw_text, entities=WANTED, score_threshold=0.4)
display(hits)
detected = set(hits.entity_type)
assert {"PERSON", "EMAIL_ADDRESS", "CUSTOMER_ID"} <= detected, detected
assert detected & {"PHONE_NUMBER", "IN_PHONE_NUMBER"}, "phone number was not detected"
assert "UK_NHS" not in detected

## 3. Measure recall on the cases you care about

Detectors have blind spots you must *measure*, not assume. Presidio's built-in `EmailRecognizer`
validates the top-level domain, so an address in a reserved TLD such as `.test` is silently
**missed**. Build a tiny labelled set, compute recall per entity, and add a recognizer when it fails.

In [ ]:
labelled = [
    {"text": "Mail me at priya.n@example.com",                    "expect": "EMAIL_ADDRESS"},
    {"text": "Mail me at priya.n@example.test",                   "expect": "EMAIL_ADDRESS"},   # reserved TLD
    {"text": "Call 9123456780 tomorrow",                          "expect": "IN_PHONE_NUMBER"},
    {"text": "Call +91-91234 56780 tomorrow",                     "expect": "IN_PHONE_NUMBER"},  # spaced format
    {"text": "Ticket for CUST-00042 escalated",                   "expect": "CUSTOMER_ID"},
    {"text": "Ticket for cust-00042 escalated",                   "expect": "CUSTOMER_ID"},      # Presidio patterns are case-insensitive
]

def recall_table(analyzer) -> pd.DataFrame:
    rows = []
    for item in labelled:
        found = {r.entity_type for r in analyzer.analyze(text=item["text"], language="en", entities=WANTED, score_threshold=0.4)}
        rows.append({**item, "detected": item["expect"] in found, "found": sorted(found)})
    return pd.DataFrame(rows)

before = recall_table(analyzer)
display(before)
print("Recall before:", f"{before.detected.mean():.0%}")

In [ ]:
# Close the gaps with a fallback e-mail recognizer (no TLD validation) and a spaced-phone pattern.
registry.add_recognizer(PatternRecognizer(
    supported_entity="EMAIL_ADDRESS", supported_language="en", name="FallbackEmailRecognizer",
    patterns=[Pattern("email-any-tld", r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", 0.6)],
))
registry.add_recognizer(PatternRecognizer(
    supported_entity="IN_PHONE_NUMBER", supported_language="en", name="InPhoneSpaced",
    patterns=[Pattern("india-mobile-spaced", r"(?<!\d)(?:\+?91[-\s]?)?[6-9]\d{4}[-\s]?\d{5}(?!\d)", 0.8)],
))
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, registry=registry, supported_languages=["en"])

after = recall_table(analyzer)
display(after)
print("Recall after:", f"{after.detected.mean():.0%}")
assert after.detected.all(), "every labelled case must now be detected"

## 4. Transform with the `AnonymizerEngine` — one operator per entity type

Different fields deserve different transformations: replace, mask (keep a prefix), hash (stable
correlation), or encrypt (reversible with a key). The choice is a *purpose* decision.

In [ ]:
analyzer_results = analyzer.analyze(text=raw_text, language="en", entities=WANTED, score_threshold=0.4)
anonymizer = AnonymizerEngine()
operators = {
    "DEFAULT":         OperatorConfig("replace", {"new_value": "<REDACTED>"}),
    "PERSON":          OperatorConfig("replace", {"new_value": "<PERSON>"}),
    "EMAIL_ADDRESS":   OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 8, "from_end": False}),
    "PHONE_NUMBER":    OperatorConfig("replace", {"new_value": "<PHONE>"}),
    "IN_PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
    "CUSTOMER_ID":     OperatorConfig("hash", {"hash_type": "sha256"}),
}
anonymized = anonymizer.anonymize(text=raw_text, analyzer_results=analyzer_results, operators=operators)
print(anonymized.text)
display(pd.DataFrame([{"entity_type": i.entity_type, "operator": i.operator, "start": i.start, "end": i.end} for i in anonymized.items]))

## 5. Purpose-specific views

The model does not need the customer's e-mail to answer a returns question. General telemetry needs
*no* identifiers. Fraud linkage needs a **stable pseudonym** — an HMAC with a managed key, so the
same customer maps to the same token without the raw ID ever leaving the trust boundary.

In [ ]:
def pseudonymize(value: str, key: bytes) -> str:
    return "psn_" + hmac.new(key, value.encode(), sha256).hexdigest()[:16]

def presidio_view(text: str, operators: dict) -> str:
    results = analyzer.analyze(text=text, language="en", entities=WANTED, score_threshold=0.4)
    return anonymizer.anonymize(text=text, analyzer_results=results, operators=operators).text

customer_id = re.search(r"CUST-\d{5}", raw_text).group(0)
views = {
    "model_input": presidio_view(raw_text, {
        "DEFAULT": OperatorConfig("replace", {"new_value": "<REDACTED>"}),
        "PERSON": OperatorConfig("replace", {"new_value": "<PERSON>"}),
        "CUSTOMER_ID": OperatorConfig("keep"),        # the model may need to reference the ticket
    }),
    "general_log": presidio_view(raw_text, {
        "DEFAULT": OperatorConfig("replace", {"new_value": "<REDACTED>"}),
    }),
    "fraud_linkage": pseudonymize(customer_id, b"synthetic-workshop-key-do-not-use"),
}
print(json.dumps(views, indent=2))

In [ ]:
# --- Contract: evidence and ordinary destinations contain no raw identifiers ------
for destination in ["model_input", "general_log"]:
    value = views[destination]
    assert "asha.rao@example.com" not in value, destination
    assert "9876543210" not in value, destination
    assert "Asha Rao" not in value, destination
assert "CUST-48291" not in views["general_log"]
assert "CUST-48291" in views["model_input"]          # deliberately kept for this purpose
assert views["fraud_linkage"].startswith("psn_")

evidence = {
    "raw_text_sha256": sha256(raw_text.encode()).hexdigest(),   # never the raw text
    "detected_entities": hits.drop(columns=["span"]).to_dict(orient="records"),
    "recall_before": float(before.detected.mean()),
    "recall_after": float(after.detected.mean()),
    "views": views,
    "controls": {
        "pre_egress_transformation": True,
        "raw_value_in_evidence": False,
        "pseudonym_key_storage": "demo only; use managed secrets in production",
    },
}
out = save_json("_evidence/05_pii_views.json", evidence)
print("PASS: raw direct identifiers are absent from exported views")
print("Wrote", out.resolve())

## What to test next

Build a labelled test set containing local names, addresses, account formats, multilingual text,
OCR errors, and adversarial separators. Track **false negatives** separately from **over-redaction**
— both cause harm: leakage on one side, unusable service (often for one language or group) on the other.
Presidio also supports the `transformers` NLP engine and multi-language registries when spaCy's small
model is not enough.